[Step 8 - FAISS]

> **MLCourse - Agentic AI - Vector Stores**
> Stage in the capstone: STORE + RETRIEVE backbone.

## What you will learn

1. Indexing the SAME Alice chunks with FAISS (`langchain-community` wrapper).
2. What `save_local` writes and why `load_local` demands an honest flag.
3. Plain `similarity_search` over a flat index.
4. Timing searches with `time.perf_counter` to feel FAISS's speed profile.
5. The real differences versus Chroma: manual persistence, no native filters.

FAISS (Facebook AI Similarity Search) is not a database - it is a library of
battle-tested ANN indexes wrapped in-process. You trade Chroma's conveniences
for raw speed and total control over serialization.

In [1]:
# ---------------------------------------------------------------------------
# Setup cell (identical in every MLCourse notebook): imports, TRACK walker,
# DATA folder creation, .env loading, matplotlib inline magic - guarded so
# the file also runs as a plain script outside Jupyter.
# ---------------------------------------------------------------------------
from pathlib import Path


def find_track(start: Path, target: str = "03_agentic_ai") -> Path:
    """Climb parent folders until a directory named ``target`` shows up."""
    for candidate in [start, *start.parents]:
        probe = candidate / target
        if probe.is_dir():
            return probe
    raise FileNotFoundError(
        f"Could not find '{target}' above {start}. "
        "Open this notebook from inside the MLCourse repository."
    )


TRACK = find_track(Path.cwd())       # .../MLCourse/03_agentic_ai
DATA = TRACK / "data"                # one shared data folder for the track
DATA.mkdir(exist_ok=True)            # no-op when it already exists

from dotenv import load_dotenv       # noqa: E402  reads KEY=value files

load_dotenv()                        # .env beside the current directory
load_dotenv(TRACK / ".env")          # .env at the track root

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass                             # magic only exists inside IPython/Jupyter

print("[setup] TRACK:", TRACK)
print("[setup] DATA :", DATA)

[setup] TRACK: D:\projects\python\MLCourse\03_agentic_ai
[setup] DATA : D:\projects\python\MLCourse\03_agentic_ai\data


## Same chunks as the Chroma notebook

Identical ingest recipe on purpose: chapter-anchored regex, 20k-char slice,
`RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)`. Keeping
the corpus constant is what makes module 08's head-to-head notebook fair -
only the STORE changes between these two notebooks.

In [2]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

ALICE_URL = "https://www.gutenberg.org/files/11/11-0.txt"


def ensure_alice(data_dir: Path) -> Path:
    """Download alice.txt ONCE into the shared data folder; reuse forever."""
    path = data_dir / "alice.txt"
    if path.exists():
        print(f"[data] cached {path.name}: {path.stat().st_size:,} bytes")
    else:
        import urllib.request
        print("[data] downloading alice.txt (one time) ...")
        urllib.request.urlretrieve(ALICE_URL, path)
        print(f"[data] saved   {path.name}: {path.stat().st_size:,} bytes")
    return path


RAW = ensure_alice(DATA).read_text(encoding="utf-8-sig")[:20_000]  # speed slice

marks = list(re.finditer(r"^CHAPTER [IVX]+\.", RAW, flags=re.MULTILINE))

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
documents = []
for i, mark in enumerate(marks):
    seg_end = marks[i + 1].start() if i + 1 < len(marks) else len(RAW)
    for piece in splitter.split_text(RAW[mark.start():seg_end]):
        documents.append(Document(
            page_content=piece,
            metadata={"source": "alice", "chapter": str(i + 1)},
        ))

print("[data] chunks built:", len(documents), "from", len(marks), "chapters")

[data] cached alice.txt: 151,191 bytes
[data] chunks built: 60 from 2 chapters


## from_documents builds a FLAT index

The default FAISS backend here is IndexFlatL2: exact brute-force L2 distance
over every vector. No approximation, perfect recall, SIMD-fast - ideal up to
roughly 100k vectors on a laptop. Bigger regimes switch to approximate index
types (IVF, HNSW), which trade a little recall for large speed wins.

In [3]:
import time                                            # perf_counter timing soon
from langchain_community.vectorstores import FAISS     # official community wrapper
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

faiss_db = FAISS.from_documents(documents, embeddings)

print("[faiss] vectors indexed :", faiss_db.index.ntotal)
print("[faiss] index type      :", type(faiss_db.index).__name__,
      "(flat = exact brute force)")

C:\Users\Thoyajaksha Kashyap\AppData\Local\Temp\ipykernel_78804\3286663518.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS     # official community wrapper


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[faiss] vectors indexed : 60
[faiss] index type      : IndexFlatL2 (flat = exact brute force)


## Persistence is YOUR job now

Chroma wrote to disk automatically. FAISS does nothing by itself:
`save_local` serializes two artifacts into a folder - `index.faiss` (the raw
vector index) and `index.pkl` (a PICKLE of the docstore holding texts +
metadata). Rerunning this cell simply overwrites; there is no server keeping
state, which is exactly why forgetting to call save loses everything.

In [4]:
SAVE_DIR = DATA / "faiss_alice"
faiss_db.save_local(str(SAVE_DIR))          # writes index.faiss + index.pkl

saved_files = sorted(p.name for p in SAVE_DIR.iterdir())
print("[faiss] saved to        :", SAVE_DIR)
print("[faiss] folder contains :", saved_files)

[faiss] saved to        : D:\projects\python\MLCourse\03_agentic_ai\data\faiss_alice
[faiss] folder contains : ['index.faiss', 'index.pkl']


## Loading back - and the scary-looking flag, explained honestly

`index.pkl` is a Python pickle. Unpickling EXECUTES ARBITRARY CODE, so loading
a pickle you did not create is genuinely dangerous - that is why LangChain
forces `allow_dangerous_deserialization=True`: the flag makes you stop and
acknowledge the risk. Chroma avoids all this (SQLite files carry no code),
which is part of why it needs no such flag.

> **Bold rule**: only load stores YOU built or that came from a source you'd
  trust with arbitrary code execution - because that is what loading grants.

In [5]:
loaded_db = FAISS.load_local(
    str(DATA / "faiss_alice"),
    embeddings,                                    # SAME model as index time
    allow_dangerous_deserialization=True,          # trusting OUR OWN files
)
print("[load] vectors loaded   :", loaded_db.index.ntotal)

for rank, doc in enumerate(loaded_db.similarity_search("a mad tea party", k=3), 1):
    snippet = doc.page_content[:70].replace("\n", " ")
    print(f"{rank}. ch.{doc.metadata['chapter']} | {snippet} ...")

[load] vectors loaded   : 60
1. ch.1 | It was all very well to say “Drink me,” but the wise little Alice was  ...
2. ch.1 | Soon her eye fell on a little glass box that was lying under the table ...
3. ch.1 | However, this bottle was _not_ marked “poison,” so Alice ventured to t ...


## Timing: feel the flat index's speed

We run every query 20 times and average with time.perf_counter - one-shot
timings on laptops are noise-dominated. Expect sub-millisecond per search at
this scale even though flat search touches EVERY vector: SIMD makes brute
force embarrassingly quick until corpora get huge.

In [6]:
QUERIES = [
    "down the rabbit hole",
    "drink me bottle",
    "pool of tears",
    "a caucus race",
    "the white rabbit's gloves",
]

REPEATS = 20
t0 = time.perf_counter()
for _ in range(REPEATS):
    for q in QUERIES:
        faiss_db.similarity_search(q, k=3)
elapsed = time.perf_counter() - t0

n_calls = REPEATS * len(QUERIES)
print("searches executed :", n_calls)
print("total wall time   : %.3f s" % elapsed)
print("avg per search    : %.3f ms" % (elapsed / n_calls * 1000))
print("(flat index scans ALL %d vectors per search - still this fast)"
      % faiss_db.index.ntotal)

searches executed : 100
total wall time   : 0.949 s
avg per search    : 9.487 ms
(flat index scans ALL 60 vectors per search - still this fast)


## FAISS vs Chroma, honestly summarized

| Concern | Chroma (notebook 01) | FAISS (this notebook) |
|---|---|---|
| Disk persistence | automatic via persist_directory | manual save_local / load_local |
| File format | SQLite (no executable content) | pickle (code execution on load) |
| Metadata filtering | where-filters inside the query | none native; filter hits yourself |
| Speed ceiling | fine for small/mid corpora | extremely fast, scales via index types |

## Takeaway

- FAISS = indexes, not storage: build fast, but persistence is on you.
- save_local writes index.faiss + index.pkl; the load flag exists because
  pickles execute code - treat store folders like programs, not data.
- Flat L2 search is exact AND blazing at course scale.

## Summary

We indexed the identical Alice chunks into a flat FAISS index, serialized it
to disk and loaded it back with full honesty about the deserialization flag,
ran plain k=3 searches, and benchmarked thousands of searches down to
fractions of a millisecond. Next: both stores side by side on identical
inputs, so the choice becomes data instead of folklore.